#### Bronze: ingest road status page
Downloads the NPS Blue Ridge Parkway road status page and stores it **unchanged**, as the raw source for the silver and gold layers. Can run several times a day; only downloads where the page actually changed are kept.

**Outputs**
- Raw HTML file, one per saved download: `/Volumes/brp/bronze/raw/road_status_page/<run_date>/page_<YYYYMMDDTHHMMSSZ>.html`
- One record per saved download: `brp.bronze.road_status_raw`

**Why both a volume and a table?**
- **Volume (landing zone):** the page exactly as received, byte for byte. The permanent original, and the source of truth if anything downstream ever needs rebuilding.
- **Bronze table:** one row per download, with metadata (time, status, fingerprint, file location) plus the HTML as text, so silver can query and parse it with SQL/Spark.
The table's `file_path` column links each row to its original file.

**Steps**
1. **Settings:** load libraries and define the source URL, storage locations, and today's run date (Eastern time)
2. **Download:** fetch the page; record the exact download time (UTC) and a SHA-256 fingerprint
3. **Gate checks:** stop before saving anything if the download looks wrong (not status 200, too small, or missing the expected headings)
4. **Check for changes:** compare the fingerprint with the most recent download in bronze; if identical, end the run without saving
5. **Save file:** write the untouched HTML to today's folder, named by download time so runs never overwrite each other
6. **Write record:** add this download to the bronze table (MERGE on source + download time, so a rerun can't create a duplicate)
7. **Audit:** confirm the table has no duplicate downloads and no unchanged pages stored back-to-back; fails the job if it does

**Safe to re-run:** an unchanged page is skipped, and rerunning the write step can't duplicate a download. Every saved download is kept; past rows are never modified.

**If a check fails:** gate failures save nothing, so just rerun or wait for the next scheduled run. For audit failures, delete the bad row by `fetched_at` in bronze.

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SETUP
# Settings and imports for the bronze step. This step downloads the NPS
# road status page and saves it exactly as received (no cleaning):
#   - the raw HTML file goes into a Unity Catalog volume
#   - one record per run goes into the bronze Delta table
# The silver notebook later reads from that table and does the parsing.
# ---------------------------------------------------------------------------


# --- Imports ---

# Standard library (built into Python)
import os                            # building and checking file paths in the volume
import hashlib                       # making a fingerprint (hash) of the page, to spot when it changes
from datetime import datetime, timezone   # current date/time; timezone.utc for UTC timestamps
from zoneinfo import ZoneInfo             # real-world time zones like "America/New_York"

# Third-party (installed packages, available on Databricks)
import requests                      # downloading the web page
from pyspark.sql import Row          # building a one-row record to append to the bronze table


# --- Settings ---

# The web page to download. Its HTML is saved unchanged; parsing happens in silver.
SOURCE_URL = "https://www.nps.gov/blri/planyourvisit/roadclosures.htm"

# Folder (a Unity Catalog volume) where each run's raw HTML file is saved.
# Format: /Volumes/<catalog>/<schema>/<volume>
VOLUME_PATH = "/Volumes/brp/bronze/raw"

# Bronze Delta table: one row per run, recording what was downloaded and when.
# Format: catalog.schema.table
BRONZE_TABLE = "brp.bronze.road_status_raw"


# --- Run date ---
# Today's date in the parkway's local time zone (Eastern), not the server's.
# Databricks clusters run on UTC, so a job running at 9 P.M. Eastern would
# otherwise be stamped with tomorrow's date.
#
# Kept in two forms:
#   run_date      -> a real date object; store THIS in the bronze table, so
#                    the column is a DATE type and matches the silver schema
#   run_date_str  -> the same date as text, "2026-11-13"; use THIS in file
#                    names, since file paths need text
run_date = datetime.now(ZoneInfo("America/New_York")).date()
run_date_str = run_date.isoformat()
print("Run date:", run_date_str)

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: DOWNLOAD THE PAGE
# Fetch the NPS road status page and keep it exactly as received. Nothing is
# cleaned or parsed here; that happens in silver. Keeping the original means
# that if a parsing rule is ever wrong, silver can be rebuilt from bronze
# without needing the page as it was on that day.
# ---------------------------------------------------------------------------


# --- Step 1: Request the page ---
response = requests.get(
    SOURCE_URL,

    # Give up after 30 seconds if NPS doesn't answer, so a slow or down site
    # fails the job quickly instead of leaving it hanging.
    timeout=30,

    # Identify this script to the website. Without a User-Agent, requests
    # sends a generic one ("python-requests/..."), which some sites block.
    # A descriptive name is also polite: the site owner can see who's calling.
    headers={"User-Agent": "brp-closures-portfolio-project"},
)

# Stop the job if NPS returned an error page (e.g. 404 Not Found, 403
# Forbidden, 500 Server Error). Without this, the error page's HTML would be
# saved to bronze as if it were the real road status page.
response.raise_for_status()


# --- Step 2: Record what was downloaded, for auditing ---

# The exact moment of the download, in UTC. UTC avoids any ambiguity
# around time zones and daylight saving changes. (run_date, from the setup
# cell, is the Eastern calendar date; this is the precise timestamp.)
fetched_at = datetime.now(timezone.utc)

# The page exactly as received, as raw bytes. .content gives the bytes
# untouched; .text would decode them into a string first, which can change
# characters if the encoding is guessed wrong.
raw_bytes = response.content

# A SHA-256 fingerprint of those bytes: a 64-character code that's identical
# for identical content and completely different if even one character
# changes. Useful for:
#   - spotting days when the page didn't change (same fingerprint as yesterday)
#   - proving a stored file hasn't been altered since it was downloaded
sha256 = hashlib.sha256(raw_bytes).hexdigest()


# --- Step 3: Print a short summary for the run log ---
print("Fetch timestamp:", fetched_at)
print("Status:", response.status_code)            # should be 200 (OK); errors stopped above
print("Size:", len(raw_bytes), "bytes")           # a sudden big change may mean a layout change
print("Fingerprint:", sha256[:16], "...")         # first 16 characters are enough to compare by eye

In [0]:
# ---------------------------------------------------------------------------
# BRONZE GATE: IS THIS DOWNLOAD WORTH SAVING?
# Checks the new download BEFORE anything is written. If any check fails,
# the job stops here, and nothing is saved to the volume or the table, so
# there's nothing to clean up afterwards.
# ---------------------------------------------------------------------------

page_text = response.text

# Gate 1: The site said OK. (raise_for_status already stops 4xx/5xx errors;
# this also catches odd replies like 204 "No Content".)
assert response.status_code == 200, f"Gate 1: Unexpected status {response.status_code}"

# Gate 2: The page isn't suspiciously small. A real road status page is
# tens of thousands of characters; an error or "maintenance" page is tiny.
# Adjust the number once you've seen a few real sizes in the review query.
assert len(raw_bytes) > 10_000, f"Gate 2: Page only {len(raw_bytes)} bytes: error page?"

# Gate 3: The page contains what silver needs to parse. If NPS redesigns the
# page, these phrases disappear, so it's better to stop now than to store a page
# that silver can't read.
assert "Sections of Parkway" in page_text, "Gate 3: State table headings missing: page layout changed?"
assert "Road status as of" in page_text, "Gate 3: 'Road status as of' line missing: page layout changed?"

print("Gate checks passed")

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SKIP IF UNCHANGED
# Compare this download with the most recent one stored in bronze. If the
# fingerprint matches, nothing new needs saving.
# The result is kept in `page_changed` (True/False) so the save and write
# cells below can check it too, which keeps manual cell-by-cell runs safe.
# ---------------------------------------------------------------------------

# Start with "nothing to compare against". This stays None on the very first
# run, when the bronze table doesn't exist yet.
last = None

if spark.catalog.tableExists(BRONZE_TABLE):
    last = spark.sql(f"""
        SELECT sha256
        FROM {BRONZE_TABLE}
        ORDER BY fetched_at DESC      -- newest download first
        LIMIT 1
    """).first()

# True  -> no previous download, or a different fingerprint: save it
# False -> same fingerprint as the last download: skip it
page_changed = not (last is not None and last["sha256"] == sha256)

if page_changed:
    print("Page is new or changed; saving.")
else:
    print("Page unchanged since last download; nothing will be saved.")
    dbutils.notebook.exit("unchanged")   # in a job or Run all, the run ends here

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SAVE THE RAW FILE
# Save the downloaded page, byte for byte, as an HTML file in the bronze
# volume. This file is the permanent original: if anything later in the
# pipeline goes wrong, it can be re-parsed from here.
#
# The file is named by its fingerprint (sha256), so identical content always
# gets the same file name. If that file already exists, it isn't written
# again. That keeps the volume free of duplicates even if a previous run
# saved the file but failed before writing its table row.
# ---------------------------------------------------------------------------

# Safety net for manual runs: if the skip cell found no change, stop here.
if not page_changed:
    dbutils.notebook.exit("unchanged")


# --- Step 1: Pick and create a folder for today ---
# One folder per calendar day (Eastern time), e.g.
#   /Volumes/brp/bronze/raw/road_status_page/2026-11-13/
folder = f"{VOLUME_PATH}/road_status_page/{run_date_str}"
os.makedirs(folder, exist_ok=True)   # no error if it already exists


# --- Step 2: Name the file after its content ---
# The first 16 characters of the fingerprint are plenty to keep names unique, e.g.
#   page_3f9a1c07b2e4d815.html
# Same content -> same name; any change to the page -> a different name.
# (The exact download time is still recorded in the table's fetched_at column.)
file_path = f"{folder}/page_{sha256[:16]}.html"


# --- Step 3: Write the file, unless this exact content is already saved ---
if os.path.exists(file_path):
    # Already saved by an earlier run (e.g. one that saved the file but failed
    # before the table write). Reuse it: the table row below will point to it.
    print("File already in volume, not rewritten:", file_path)
else:
    # "wb" = binary mode: bytes saved exactly as downloaded, no conversion.
    with open(file_path, "wb") as f:
        f.write(raw_bytes)
    print("Saved to:", file_path)

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: SAVE THE RAW FILE
# Save the downloaded page, byte for byte, as an HTML file in the bronze
# volume. This file is the permanent original: if anything later in the
# pipeline goes wrong, it can be re-parsed from here.
#
# The job can run several times a day, so every download gets its own file
# and nothing is ever overwritten. (Unchanged pages were already skipped by
# the previous cell, so only new versions of the page reach this point.)
# ---------------------------------------------------------------------------


# --- Step 1: Pick and create a folder for today ---
# One folder per calendar day (Eastern time), so files are easy to browse by date:
#   /Volumes/brp/bronze/raw/road_status_page/2026-11-13/
# run_date_str is the text form of the date, set in the setup cell.
folder = f"{VOLUME_PATH}/road_status_page/{run_date_str}"

# Create the folder, plus any missing parent folders.
# exist_ok=True means "no error if it already exists", which is the normal
# case for the second and later runs of the day.
os.makedirs(folder, exist_ok=True)


# --- Step 2: Name the file after the exact download time ---
# fetched_at (from the download cell) is the UTC moment the page was fetched.
# Formatting it into the name gives every run a unique file, e.g.
#   page_20261113T191502Z.html  ->  2026-11-13 at 19:15:02 UTC
#   %Y%m%d = date, T = separator, %H%M%S = time, Z = "this time is UTC"
# Files named this way also sort in time order automatically.
file_path = f"{folder}/page_{fetched_at:%Y%m%dT%H%M%SZ}.html"


# --- Step 3: Write the file ---
# "wb" = write in binary mode, so the bytes are saved exactly as downloaded,
# with no text conversion. The `with` block closes the file automatically,
# even if an error happens partway through.
with open(file_path, "wb") as f:
    f.write(raw_bytes)

print("Saved to:", file_path)

In [0]:
# ---------------------------------------------------------------------------
# BRONZE INGEST: RECORD THIS DOWNLOAD IN THE BRONZE TABLE
# Add one row describing this download to the bronze Delta table. The table
# keeps a full history: one row per download, several per day if the job
# runs several times. The silver notebook reads the newest row from here.
# ---------------------------------------------------------------------------


# --- Step 1: Build the record for this download ---
record = Row(
    source="road_status_page",         # which source this is (room for more sources later)
    url=SOURCE_URL,                    # the exact page that was downloaded
    run_date=run_date,                 # Eastern calendar date (a real date), for grouping by day
    fetched_at=fetched_at,             # exact UTC moment of the download: unique per run
    http_status=response.status_code,  # the site's response code (200 = OK)
    sha256=sha256,                     # fingerprint of the page, used to skip unchanged downloads
    file_path=file_path,               # where the raw HTML file was saved in the volume
    raw_html=response.text,            # the page as text, ready for silver to parse
                                       # (the untouched bytes are in the file above)
)

# Turn the record into a one-row Spark DataFrame so it can be written to a table.
df = spark.createDataFrame([record])


# --- Step 2: Write it to the bronze table ---
if not spark.catalog.tableExists(BRONZE_TABLE):
    # First run ever: the table doesn't exist yet, so create it from this row.
    # Its columns and types are taken from the record above.
    df.write.saveAsTable(BRONZE_TABLE)
    print("Created", BRONZE_TABLE, "with the first download")

else:
    # Normal run: add the row, but only if this exact download isn't already
    # there. MERGE needs a named view to read from, so register the DataFrame
    # under the temporary name "incoming".
    df.createOrReplaceTempView("incoming")

    # Match on (source, fetched_at), which identifies one specific download.
    #   - Not in the table yet -> insert it (the normal case).
    #   - Already there        -> do nothing. This only happens if this cell
    #     is rerun by hand, and it stops that from creating a duplicate row.
    spark.sql(f"""
        MERGE INTO {BRONZE_TABLE} AS t
        USING incoming AS s
        ON  t.source     = s.source
        AND t.fetched_at = s.fetched_at
        AND t.sha256     = s.sha256
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("Added this download to", BRONZE_TABLE)

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- BRONZE REVIEW: WHAT'S BEEN DOWNLOADED
-- Lists every download stored in the bronze table, newest first. Use it after
-- a run to confirm the latest download landed and looks reasonable.
--
-- What to look for:
--   - The top row's fetched_at_eastern should match the run you just did.
--   - http_status should be 200 on every row.
--   - html_length should be roughly the same from row to row. A sudden big
--     drop (or near 0) usually means an error page or a site redesign.
--   - Rows on the same day with different fingerprints mean the page really
--     changed between runs (identical pages are skipped, so they never appear).
-- ---------------------------------------------------------------------------
SELECT
  run_date,                                         -- pipeline day (America/New_York); several rows can share one
  fetched_at,                                       -- exact download time (UTC)
  from_utc_timestamp(fetched_at, 'America/New_York')
    AS fetched_at_eastern,                          -- same moment in Eastern time, easier to read
  http_status,                                      -- server reply code; 200 means OK
  left(sha256, 16) AS sha256_short,                 -- first 16 characters of the fingerprint; enough to compare by eye
  length(raw_html) AS html_length,                  -- page size in characters
  file_path                                         -- where the raw HTML file is saved, for opening it if needed
FROM brp.bronze.road_status_raw
ORDER BY fetched_at DESC                            -- newest download at the top (was run_date, which ties on same-day runs)

In [0]:

%sql
-- ---------------------------------------------------------------------------
-- MAINTENANCE: DELETE THE NEWEST DOWNLOAD  (commented out on purpose)
--
-- When to use: a test run or a bad download got saved and you want to
-- remove it. This deletes whichever download is newest at the moment
-- you run it.
--
-- How to use:
--   1. Run the PREVIEW query on its own and confirm it's the row you mean.
--   2. Uncomment the two DELETE statements (silver first, then bronze).
--   3. Run the cell ONCE.
--   4. Comment the DELETE statements out again.
--
-- Warnings:
--   - Each run deletes the newest row at that moment, so running it twice
--     deletes TWO downloads (the second one may be real data).
--   - The raw HTML file in the volume is NOT deleted; remove it by hand
--     from Catalog -> brp -> bronze -> raw if needed.
--   - Deleted by mistake? Undo with:
--       DESCRIBE HISTORY brp.bronze.road_status_raw;            -- find the version before the delete
--       RESTORE TABLE brp.bronze.road_status_raw TO VERSION AS OF <n>;
--     (same for the silver table)
-- ---------------------------------------------------------------------------

-- PREVIEW: the download that would be deleted
-- SELECT fetched_at, left(sha256, 16) AS sha256_short, file_path
-- FROM brp.bronze.road_status_raw
-- WHERE fetched_at = (SELECT MAX(fetched_at) FROM brp.bronze.road_status_raw);

-- Remove the newest download itself
-- DELETE FROM brp.bronze.road_status_raw
-- WHERE fetched_at = (SELECT MAX(fetched_at) FROM brp.bronze.road_status_raw);

In [0]:
# ---------------------------------------------------------------------------
# BRONZE AUDIT: IS THE TABLE STILL IN GOOD SHAPE?
# Runs after the write. These problems should already be prevented upstream;
# this confirms it. Any failure stops the job, so silver doesn't run on bad data.
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# BRONZE CHECK 1: NO DUPLICATE DOWNLOADS
# Each download is identified by its source + exact download time
# (fetched_at). Several downloads per day are normal, but the SAME download
# should never be stored twice. That would mean the write cell was rerun
# and the MERGE didn't catch it.
#
# Expected result: NO ROWS. Any row returned is a duplicate to investigate.
# ---------------------------------------------------------------------------
dupes = spark.sql(f"""
    SELECT source, fetched_at, COUNT(*) AS copies
    FROM {BRONZE_TABLE}
    GROUP BY source, fetched_at
    HAVING COUNT(*) > 1
""").collect()
assert not dupes, f"Audit 1: Duplicate downloads found: {dupes}"


# ---------------------------------------------------------------------------
# BRONZE CHECK 2: NO UNCHANGED PAGE STORED TWICE IN A ROW
# The ingest skips a download when its fingerprint (sha256) matches the
# previous one, so two back-to-back rows should never share a fingerprint.
# If any do, the "skip if unchanged" cell isn't working.

# (The same fingerprint CAN appear again later, e.g. a road closes and then
# reopens, putting the page back exactly as before. That's fine, which is
# why this only compares each row with the one right before it.)

# Expected result: NO ROWS.
# ---------------------------------------------------------------------------
repeats = spark.sql(f"""
    WITH ordered AS (
      SELECT source, fetched_at, sha256,
             LAG(sha256) OVER (PARTITION BY source ORDER BY fetched_at) AS previous_sha256
      FROM {BRONZE_TABLE}
    )
    SELECT source, fetched_at FROM ordered WHERE sha256 = previous_sha256
""").collect()
assert not repeats, f"Audit 2: Unchanged pages stored back-to-back: {repeats}"

print("Bronze audit passed")